<div dir=rtl>

# مینی‌پروژه: یادگیری رابطه‌های غیرخطی با شبکه‌ی MLP

این Notebook برای شروع پروژه آماده شده است. سلول‌های ابتدایی، کتابخانه‌ها را نصب و وارد می‌کنند، فایل مرجع ۱۰۰۰ نمونه‌ای را می‌خوانند و با استفاده از شماره‌ی دانشجویی شما، فایل CSV اختصاصی شامل ۱۰۰ نمونه را می‌سازند. **بقیه‌ی مراحل پروژه را خودتان مطابق صورت سؤال انجام دهید.**

## پیش از اجرا: فایل‌ها را کنار هم قرار دهید

۱. فایل `proj1-q.ipynb` و فایل `master_nonlinear_mlp_dataset_1000.csv` را در **یک پوشه** قرار دهید. نام فایل CSV را تغییر ندهید.

۲. اگر از Jupyter Notebook یا VS Code استفاده می‌کنید، Notebook را از همان پوشه باز و اجرا کنید. اگر خطای پیدا نشدن فایل دریافت کردید، مسیر کاری Notebook را بررسی کنید.

۳. اگر از Google Colab استفاده می‌کنید، فایل CSV را هنگام درخواست بارگذاری کنید.

۴. فقط شماره‌ی دانشجویی خودتان را (برای کار انفرادی) یا شماره‌های هر دو عضو گروه را (برای کار دونفره) در سلول seed وارد کنید. فایل مرجع مشترک را ویرایش نکنید.

فایل خروجی گروه با نامی مانند `nonlinear_mlp_data_seed_1234.csv` ذخیره می‌شود و در ادامه‌ی Notebook به‌عنوان داده‌ی اصلی پروژه بارگذاری خواهد شد. فایل مرجع شامل ۱۰۰۰ نمونه است؛ فایل اختصاصی شما شامل ۱۰۰ نمونه از همان مجموعه است.

</div>

<div dir=rtl>

## ۱. نصب کتابخانه‌های موردنیاز

این سلول را یک بار اجرا کنید. نصب TensorFlow ممکن است کمی زمان ببرد. در صورت نصب بودن کتابخانه‌ها، پیام‌های مربوط به نصب مجدد را می‌توانید نادیده بگیرید.
</div>

In [6]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow pathlib

Note: you may need to restart the kernel to use updated packages.


<div dir=rtl>

## ۲. وارد کردن کتابخانه‌ها

کتابخانه‌های اصلی پروژه از ابتدا وارد شده‌اند؛ در صورت نیاز می‌توانید در ادامه کتابخانه‌های دیگری نیز اضافه کنید.
</div>

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.22.0-rc0


<div dir=rtl>

## ۳. ساخت seed از شماره‌ی دانشجویی

سلول بعدی شماره‌ی دانشجویی را از شما می‌پرسد. شماره‌ی کامل یا فقط چهار رقم آخر قابل قبول است. برای گروه دونفره، شماره‌ی عضو دوم را نیز وارد کنید؛ در کار انفرادی، ورودی عضو دوم را خالی بگذارید. کد چهار رقم آخر را استخراج، برای گروه دونفره آن‌ها را از کوچک به بزرگ مرتب و به هم متصل می‌کند.

همین seed برای انتخاب فایل ۱۰۰‌نمونه‌ای و برای تقسیم Training/Validation/Test در ادامه‌ی پروژه استفاده خواهد شد. با وارد کردن همان شماره‌ها، انتخاب داده‌ها قابل تکرار می‌ماند.
</div>

In [1]:
def last_four_digits(student_id):
    # پشتیبانی از رقم‌های فارسی و عربی در ورودی شماره دانشجویی
    student_id = str(student_id).translate(str.maketrans(
        '۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789'
    )).strip()
    digits = ''.join(ch for ch in student_id if ch.isdigit())
    if len(digits) < 4:
        raise ValueError('شماره دانشجویی باید دست‌کم چهار رقم داشته باشد.')
    return digits[-4:].zfill(4)

student_id_1 = '12345'
student_id_2 = '67891'

if not student_id_1:
    raise ValueError('وارد کردن شماره دانشجویی عضو اول الزامی است.')

student_ids = [student_id_1]
if student_id_2:
    student_ids.append(student_id_2)
if len(student_ids) > 2:
    raise ValueError('این پروژه برای یک نفر یا یک گروه دونفره طراحی شده است.')

suffixes = sorted(last_four_digits(sid) for sid in student_ids)
SEED_STRING = ''.join(suffixes)
SEED = int(SEED_STRING)

print('چهار رقم آخر مرتب‌شده:', suffixes)
print('SEED:', SEED)

چهار رقم آخر مرتب‌شده: ['1234', '6789']
SEED: 12346789


<div dir=rtl>

## ۴. انتخاب ۱۰۰ نمونه از فایل مرجع ۱۰۰۰‌نمونه‌ای

روش انتخاب طوری طراحی شده که ۱۰۰ نمونه فقط از ابتدای فایل برداشته نشوند. با توجه به ساختار فایل مرجع، در فایل هر گروه از هر دهکِ `x`، `y` و `z` دقیقاً ۱۰ نمونه انتخاب می‌شود. انتخاب با seed شما قابل تکرار است و ردیف‌ها بدون جایگذاری انتخاب می‌شوند.

اگر CSV در پوشه‌ی کاری پیدا نشود، در Google Colab پنجره‌ی بارگذاری فایل باز می‌شود. فایل `master_nonlinear_mlp_dataset_1000.csv` را انتخاب کنید.
</div>

In [ ]:
MASTER_FILENAME = 'master_nonlinear_mlp_dataset_1000.csv'
MASTER_PATH = Path(MASTER_FILENAME)

if not MASTER_PATH.exists():
    try:
        from google.colab import files
        print(f'لطفاً فایل {MASTER_FILENAME} را انتخاب کنید.')
        files.upload()
    except ImportError:
        pass

if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f'فایل {MASTER_FILENAME} پیدا نشد. آن را کنار Notebook قرار دهید. '
        f'پوشه کاری فعلی: {Path.cwd()}'
    )

master_df = pd.read_csv(MASTER_PATH)
expected_columns = ['x', 'y', 'z', 'u']
if master_df.shape != (1000, 4) or master_df.columns.tolist() != expected_columns:
    raise ValueError('فایل مرجع باید دقیقاً ۱۰۰۰ سطر و ستون‌های x، y، z و u داشته باشد.')
if master_df.isna().any().any():
    raise ValueError('فایل مرجع دارای مقدار گمشده است؛ نسخه اصلی را دوباره دریافت کنید.')

# این بازه‌ها با فایل مرجع ارائه‌شده هماهنگ هستند.
N_BINS = 10
FEATURE_RANGES = {
    'x': (0.5, 3.5),
    'y': (0.5, 3.5),
    'z': (-2.0, 6.0),
}

def get_bin_indices(data, column, value_range):
    low, high = value_range
    values = data[column].to_numpy()
    bins = np.floor((values - low) / (high - low) * N_BINS).astype(int)
    return np.clip(bins, 0, N_BINS - 1)

master_bins = {
    column: get_bin_indices(master_df, column, FEATURE_RANGES[column])
    for column in ('x', 'y', 'z')
}

lookup = {}
for row_index, key in enumerate(zip(master_bins['x'], master_bins['y'], master_bins['z'])):
    key = tuple(int(v) for v in key)
    if key in lookup:
        raise ValueError('ساختار فایل مرجع با نمونه‌برداری طبقه‌بندی‌شده سازگار نیست؛ فایل اصلی را دوباره دریافت کنید.')
    lookup[key] = row_index
if len(lookup) != 1000:
    raise ValueError('فایل مرجع کامل نیست یا ساختار مورد انتظار را ندارد.')

rng = np.random.default_rng(SEED)
x_order = rng.permutation(N_BINS)
y_order = rng.permutation(N_BINS)
z_order = rng.permutation(N_BINS)
offset = int(rng.integers(0, N_BINS))
selected_indices = []
for i, x_bin in enumerate(x_order):
    for j, y_bin in enumerate(y_order):
        z_bin = z_order[(i + j + offset) % N_BINS]
        selected_indices.append(lookup[(int(x_bin), int(y_bin), int(z_bin))])

selected_indices = np.asarray(selected_indices, dtype=int)
selected_indices = selected_indices[rng.permutation(len(selected_indices))]
group_df = master_df.iloc[selected_indices][expected_columns].reset_index(drop=True)

# بررسی می‌کنیم که دقیقاً ۱۰۰ ردیف بدون تکرار و با پوشش متوازن انتخاب شده باشد.
assert len(group_df) == 100
assert not group_df.duplicated().any()
for column in ('x', 'y', 'z'):
    bins = get_bin_indices(group_df, column, FEATURE_RANGES[column])
    counts = np.bincount(bins, minlength=N_BINS)
    assert np.all(counts == 10), f'نمونه‌برداری نامتوازن برای {column}: {counts}'

GROUP_CSV = Path(f'nonlinear_mlp_data_seed_{SEED}.csv')
group_df.to_csv(GROUP_CSV, index=False, float_format='%.8f')

print('تعداد نمونه‌های فایل مرجع:', len(master_df))
print('تعداد نمونه‌های انتخاب‌شده:', len(group_df))
print('فایل اختصاصی شما ذخیره شد:', GROUP_CSV.resolve())
print('ستون‌ها:', group_df.columns.tolist())
print('تعداد نمونه در هر دهک از x، y و z برابر ۱۰ است.')

<div dir=rtl>

## ۵. بارگذاری فایل اختصاصی گروه به‌عنوان داده‌ی پروژه

از این نقطه به بعد، متغیر `df` همان فایل ۱۰۰‌نمونه‌ای شماست. در تقسیم داده نیز از `SEED` به‌عنوان `random_state` استفاده کنید تا آزمایش‌ها تکرارپذیر باشند.
</div>

In [10]:
df = pd.read_csv(GROUP_CSV)

print('فایل داده‌ی مورد استفاده:', GROUP_CSV.name)
print('ابعاد داده:', df.shape)
display(df.head())

فایل داده‌ی مورد استفاده: nonlinear_mlp_data_seed_12346789.csv
ابعاد داده: (100, 4)


,x,y,z,u
0,3.405522,2.490246,2.118478,36.024381
1,3.036354,1.083519,5.215413,27.207965
2,3.475541,2.070494,5.602867,39.425039
3,1.109826,3.043664,4.834509,15.964188
4,2.340083,1.438381,-0.301781,13.617534


<div dir=rtl>

---
## از اینجا به بعد با شماست

از اینجا به بعد، بخش‌های پروژه را خودتان مطابق صورت سؤال بنویسید: بررسی داده، تحلیل اکتشافی و نمودارها، تقسیم داده، مقیاس‌بندی، طراحی و مقایسه‌ی حداقل پنج پیکربندی MLP، ارزیابی نهایی و نتیجه‌گیری.

یادتان باشد انتخاب مدل با Validation انجام می‌شود؛ داده‌ی Test را فقط پس از انتخاب مدل نهایی ارزیابی کنید. در زمان تقسیم داده از `random_state=SEED` استفاده کنید.
</div>